In [ ]:
import pandas as pd
import os

# ============================================================
# 1. KHAI BÁO ĐƯỜNG DẪN FILE NGUỒN
# ============================================================

promotion_path = "/content/promotion_new.csv"


# ============================================================
# 2. KIỂM TRA FILE CÓ TỒN TẠI HAY KHÔNG
# ============================================================

if not os.path.exists(promotion_path):
    raise FileNotFoundError("Không tìm thấy file promotion_new.csv trong /content/")


# ============================================================
# 3. ĐỌC DỮ LIỆU TỪ CSV
# ============================================================

promotion_df = pd.read_csv(promotion_path)


# ============================================================
# 4. HÀM CHUẨN HÓA TÊN CỘT
# Ví dụ:
# "Product ID"  -> "product_id"
# " PRICE "     -> "price"
# ============================================================

def normalize_columns(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()                     # Xóa khoảng trắng đầu/cuối
        .str.lower()                     # Chuyển về chữ thường
        .str.replace(" ", "_", regex=False)  # Khoảng trắng -> _
    )

    return df


promotion_df = normalize_columns(promotion_df)


# ============================================================
# 5. TẠO DIM_PROMOTION
#
# Star Schema:
# promo_key          : Surrogate Key
# promo_id           : Business Key từ hệ thống nguồn
# promo_name
# promo_type
# discount_value
# promo_channel
# app_category
# stackable_flag
# ============================================================

required_promotion_columns = [
    "promo_id",
    "promo_name",
    "promo_type",
    "discount_value",
    "promo_channel",
    "app_category",
    "stackable_flag"
]


# Kiểm tra các cột cần thiết
missing_promotion_columns = [
    col for col in required_promotion_columns
    if col not in promotion_df.columns
]

if missing_promotion_columns:
    raise ValueError(
        f"promotion_new.csv đang thiếu các cột: {missing_promotion_columns}"
    )


# Chỉ lấy các cột cần thiết
dim_promotion = promotion_df[required_promotion_columns].copy()


# ------------------------------------------------------------
# Làm sạch PROMOTION
# ------------------------------------------------------------

# Loại bỏ promotion không có promo_id
dim_promotion = dim_promotion.dropna(subset=["promo_id"])


# Các cột dạng text
promotion_text_columns = [
    "promo_id",
    "promo_name",
    "promo_type",
    "promo_channel",
    "app_category"
]

for col in promotion_text_columns:
    dim_promotion[col] = (
        dim_promotion[col]
        .astype("string")
        .str.strip()
    )


# Chuyển discount_value sang kiểu số
dim_promotion["discount_value"] = pd.to_numeric(
    dim_promotion["discount_value"],
    errors="coerce"
)


# ============================================================
# 6. CHUẨN HÓA STACKABLE_FLAG THÀNH BOOLEAN
#
# Các dạng như:
# TRUE / True / true / 1 / Yes / Y
# sẽ được chuyển thành True
#
# FALSE / False / false / 0 / No / N
# sẽ được chuyển thành False
# ============================================================

def convert_boolean(value):

    # Nếu dữ liệu bị thiếu
    if pd.isna(value):
        return False

    value = str(value).strip().lower()

    true_values = ["true", "1", "yes", "y", "t"]
    false_values = ["false", "0", "no", "n", "f"]

    if value in true_values:
        return True

    if value in false_values:
        return False

    # Nếu gặp giá trị lạ thì mặc định False
    return False


dim_promotion["stackable_flag"] = (
    dim_promotion["stackable_flag"]
    .apply(convert_boolean)
)


# Nếu một promo_id xuất hiện nhiều lần,
# chỉ giữ bản ghi cuối cùng
dim_promotion = (
    dim_promotion
    .drop_duplicates(subset=["promo_id"], keep="last")
    .reset_index(drop=True)
)


# ============================================================
# 7. TẠO SURROGATE KEY CHO DIM_PROMOTION
# ============================================================

dim_promotion.insert(
    0,
    "promo_key",
    range(1, len(dim_promotion) + 1)
)


# Sắp xếp đúng thứ tự cột trong Star Schema
dim_promotion = dim_promotion[
    [
        "promo_key",
        "promo_id",
        "promo_name",
        "promo_type",
        "discount_value",
        "promo_channel",
        "app_category",
        "stackable_flag"
    ]
]


# ============================================================
# 8. XUẤT FILE DIM_PROMOTION
# ============================================================

dim_promotion.to_csv(
    "/content/dim_promotion.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. THÔNG BÁO HOÀN THÀNH
# ============================================================

print("ETL hoàn tất!")

print("\nĐã tạo:")
print("/content/dim_promotion.csv")